# Extras for the web app: (A) full R3D-18 dual-stream evaluation, (B) parity sample pack
### Capstone C -- no training, ~20-30 min on one GPU

**(A)** The Capstone B dual-stream model (`r3d_18` + geometry Transformer, 1000 classes) only has
recorded top-1/top-5. This re-evaluates its saved `best_model.pt` on the same 12,560-clip test split
to get **precision / recall / F1, confusion matrix, per-class metrics and CPU latency** -- the same
artifacts the other three 1000-class models already have -- and exports `app_export/`.
Preprocessing is copied from that notebook (`/255`, Kinetics mean/std, no TTA). The reproduced
top-1 is checked against the recorded 0.282.

**(B)** Copies a small set of **raw test-split videos together with their cached `.npz`** so the web
app's video -> landmarks / signer-crop pipeline can be verified offline against exactly what the
models were trained on. (These files stay private to the team; they are not published in the app.)

### How to run on Kaggle
1. *File -> Import Notebook* -> upload this `.ipynb`.
2. **Add Input**: the 3 `asl-shared-cache-building-id0x` outputs, the `asl-citizen` dataset
   (needs `splits/` **and** `videos/`), and **the output of the previous r3d_18 notebook**
   (the one with `checkpoints/pilot_1000cls/best_model.pt`). Paths are auto-discovered.
3. Accelerator = **GPU T4 x1** (or P100). Internet not required.
4. **Save Version -> Save & Run All**, then download `extras/` from the Output tab.

## 1. Setup

In [ ]:
import csv
import glob
import hashlib
import json
import math
import os
import random
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from IPython.display import display
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

NOTEBOOK_START = time.time()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_GPUS = torch.cuda.device_count()


def _env(name, default):
    # Environment overrides exist ONLY for the local CPU smoke test of this notebook.
    # On Kaggle none of these are set, so the defaults in CONFIG below are what run.
    return os.environ.get(name, default)


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


print(f"Torch {torch.__version__} | device={DEVICE} | GPUs={N_GPUS}")
for i in range(N_GPUS):
    print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")

## 2. Configuration

In [ ]:
class CONFIG:
    WORK_DIR = Path(_env("ASL_WORK_DIR", "/kaggle/working"))
    CACHE_DIRS = [
        Path("/kaggle/input/notebooks/aniruddha2022/asl-shared-cache-building-id00/feature_cache_shard_0"),
        Path("/kaggle/input/notebooks/redoanmozumder/asl-shared-cache-building-id02/feature_cache_shard_2"),
        Path("/kaggle/input/notebooks/pranaysahaps/asl-shared-cache-building-id01/feature_cache_shard_1"),
    ]
    SPLITS_DIR = Path("/kaggle/input/datasets/pranaysahaps/asl-citizen/splits")
    OUT_DIR = WORK_DIR / "extras"
    TOP_K_CLASSES = int(_env("ASL_TOP_K", 1000))
    EXPECTED_FINGERPRINT = _env("ASL_EXPECTED_FP", "72d3ef78291c")   # the 1000-class list all models share
    R3D_CHECKPOINT = None            # auto-discovered: /kaggle/input/**/pilot_1000cls/best_model.pt
    RECORDED_TEST_TOP1 = 0.282       # from the r3d_18 notebook's own test cell (no TTA)
    N_SAMPLE_VIDEOS = int(_env("ASL_N_SAMPLES", 30))
    BATCH_SIZE = 32
    NUM_WORKERS = int(_env("ASL_NUM_WORKERS", 4))
    SMOKE_TEST = _env("ASL_SMOKE_TEST", "0") == "1"
    SMOKE_SAMPLES = {"train": 16, "val": 16, "test": 64}
    SEED = 42


def resolve_inputs():
    """Use CONFIG paths if they exist; otherwise auto-discover the cache shards / splits
    under /kaggle/input (Kaggle mount paths change depending on how inputs are attached)."""
    env_cache = os.environ.get("ASL_CACHE_DIRS")
    if env_cache:
        CONFIG.CACHE_DIRS = [Path(p) for p in env_cache.split(os.pathsep)]
    env_splits = os.environ.get("ASL_SPLITS_DIR")
    if env_splits:
        CONFIG.SPLITS_DIR = Path(env_splits)

    if not all(p.exists() for p in CONFIG.CACHE_DIRS):
        found = sorted({Path(p) for p in glob.glob("/kaggle/input/**/feature_cache_shard_*", recursive=True)
                        if Path(p).is_dir()})
        print(f"[inputs] configured CACHE_DIRS not all present -- auto-discovered {len(found)} shard dirs:")
        for p in found:
            print("   ", p)
        CONFIG.CACHE_DIRS = found
    assert CONFIG.CACHE_DIRS, ("No feature_cache_shard_* directories found. Attach the three "
                               "asl-shared-cache-building notebooks' outputs as inputs (see top of notebook).")
    if len(CONFIG.CACHE_DIRS) != 3:
        print(f"[inputs] WARNING: expected 3 cache shards (as in the v10 notebooks), found "
              f"{len(CONFIG.CACHE_DIRS)}. The class subset may differ from the v10 runs.")

    if not (CONFIG.SPLITS_DIR / "train.csv").exists():
        hits = sorted(glob.glob("/kaggle/input/**/splits/train.csv", recursive=True))
        assert hits, "ASL Citizen splits/train.csv not found -- attach the asl-citizen dataset."
        CONFIG.SPLITS_DIR = Path(hits[0]).parent
        print(f"[inputs] auto-discovered SPLITS_DIR: {CONFIG.SPLITS_DIR}")
    print("Cache shards:", *CONFIG.CACHE_DIRS, sep="\n   ")
    print("Splits dir:  ", CONFIG.SPLITS_DIR)


resolve_inputs()
seed_everything(CONFIG.SEED)
CONFIG.OUT_DIR.mkdir(parents=True, exist_ok=True)


VIDEOS_DIR = CONFIG.SPLITS_DIR.parent / "videos"
if os.environ.get("ASL_VIDEOS_DIR"):
    VIDEOS_DIR = Path(os.environ["ASL_VIDEOS_DIR"])
if not VIDEOS_DIR.exists():
    hits = sorted(glob.glob("/kaggle/input/**/asl-citizen/videos", recursive=True))
    VIDEOS_DIR = Path(hits[0]) if hits else VIDEOS_DIR
print("Videos dir:  ", VIDEOS_DIR, "(found)" if VIDEOS_DIR.exists() else "(NOT FOUND -- part B will be skipped)")

## Build the manifest & apply the Top-1000 class filter

**Copied verbatim** from the team's v10 notebook (Section 3): rebuild the label space from the
official ASL Citizen `train.csv` / `val.csv` / `test.csv`, restricted to videos present in the
cache, then keep the `TOP_K_CLASSES` most-represented classes (remapped to `0..K-1`, most
represented = index 0). Same code + same cache = same 1000 classes and same index mapping as the
team's other 1000-class runs, so every model in the comparison is evaluated on the identical
signer-independent test split.

The class list is saved to `class_list.json` with a fingerprint -- both notebooks print it, and
**the fingerprints must match** for the comparison to be valid.

In [ ]:
cache_index = {}
for shard_dir in CONFIG.CACHE_DIRS:
    assert shard_dir.exists(), f"Cache shard not found: {shard_dir} -- update CONFIG.CACHE_DIRS"
    for f in shard_dir.glob("*.npz"):
        cache_index[f.stem] = f
print(f"Total cached .npz files across {len(CONFIG.CACHE_DIRS)} shards: {len(cache_index)}")

split_files = {
    "train": CONFIG.SPLITS_DIR / "train.csv",
    "val": CONFIG.SPLITS_DIR / "val.csv",
    "test": CONFIG.SPLITS_DIR / "test.csv",
}
rows, all_glosses = [], set()
for subset, csv_path in split_files.items():
    assert csv_path.exists(), f"Missing split file: {csv_path}"
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            video_id = Path(row["Video file"]).stem
            gloss = row["Gloss"]
            all_glosses.add(gloss)
            if video_id in cache_index:
                rows.append((video_id, gloss, subset))

class_list_full = sorted(all_glosses)
class_to_idx_full = {g: i for i, g in enumerate(class_list_full)}

nslt_data_full = {}
for video_id, gloss, subset in rows:
    nslt_data_full[video_id] = {"action": [class_to_idx_full[gloss], 0, 0], "subset": subset}

print(f"Cached rows matched against splits: {len(rows)}")
print(f"Full ASL Citizen vocabulary present in cache: {len(class_list_full)} classes")

# --- top-K filter, identical remap logic to the v10 notebooks ---
class_video_counts = Counter(v["action"][0] for v in nslt_data_full.values())
top_k_old_indices = [c for c, _ in class_video_counts.most_common(CONFIG.TOP_K_CLASSES)]
top_k_old_indices_set = set(top_k_old_indices)
old_to_new_idx = {old_idx: new_idx for new_idx, old_idx in enumerate(top_k_old_indices)}

nslt_data = {
    vid: {"action": [old_to_new_idx[meta["action"][0]], 0, 0], "subset": meta["subset"]}
    for vid, meta in nslt_data_full.items()
    if meta["action"][0] in top_k_old_indices_set
}
class_list = {old_to_new_idx[old]: class_list_full[old] for old in top_k_old_indices}
NUM_CLASSES = len(class_list)

subset_counts = Counter(v["subset"] for v in nslt_data.values())
print(f"\nTop-{CONFIG.TOP_K_CLASSES} filter applied -- NUM_CLASSES: {NUM_CLASSES}")
print(f"Videos remaining: {len(nslt_data)} across splits {dict(subset_counts)}")

per_class_counts = Counter(v["action"][0] for v in nslt_data.values())
print(f"Videos/class (post-filter, all splits combined): "
      f"min={min(per_class_counts.values())} median={int(np.median(list(per_class_counts.values())))} "
      f"max={max(per_class_counts.values())}")

# --- added: persist the exact class mapping for the web app + cross-notebook check ---
CLASS_NAMES = [class_list[i] for i in range(NUM_CLASSES)]
CLASS_FINGERPRINT = hashlib.sha1("\n".join(CLASS_NAMES).encode("utf-8")).hexdigest()[:12]
with open(CONFIG.OUT_DIR / "class_list.json", "w", encoding="utf-8") as f:
    json.dump({"num_classes": NUM_CLASSES, "fingerprint": CLASS_FINGERPRINT,
               "rule": f"ASL Citizen top-{CONFIG.TOP_K_CLASSES} most-represented glosses in cache "
                       f"(Counter.most_common), index 0 = most represented",
               "classes": CLASS_NAMES}, f, indent=1)
print(f"\nClass-list fingerprint: {CLASS_FINGERPRINT}   (must match across all 1000-class notebooks)")


def split_samples(subset):
    # Same sample construction (and order) as the v10 WLASLCachedDataset.
    s = [(vid, meta["action"][0]) for vid, meta in nslt_data.items()
         if meta["subset"] == subset and vid in cache_index]
    if CONFIG.SMOKE_TEST:
        rng = random.Random(0)
        s = rng.sample(s, min(len(s), CONFIG.SMOKE_SAMPLES[subset]))
    return s


train_samples, val_samples, test_samples = (split_samples(s) for s in ("train", "val", "test"))
print(f"Train / val / test samples: {len(train_samples)} / {len(val_samples)} / {len(test_samples)}"
      + ("   [SMOKE TEST SUBSET]" if CONFIG.SMOKE_TEST else ""))


assert CLASS_FINGERPRINT == CONFIG.EXPECTED_FINGERPRINT, (
    f"class list {CLASS_FINGERPRINT} != expected {CONFIG.EXPECTED_FINGERPRINT}: inputs differ from the other runs")
print("Class list matches the other 1000-class models.")

## 3. Model definitions (identical to the web app's `inference/architectures.py`)

In [ ]:
"""Exact model definitions for every trained ASL Vision checkpoint.

Each class reproduces the training notebook's module tree, so state_dict keys match and
checkpoints load with strict=True. Only inference-irrelevant training details (backbone
freezing, pretrained-weight download) are omitted.

Sources:
  GeometryOnlySLRModel  -- asl-*-class-single-stream-model-pilot-run.ipynb (v10), Section 5
  DualStreamSLRModel    -- previous_r3d_18 notebook.ipynb (v9), Section 9
  build_swin3d_t        -- training_notebooks/train_1000_swin3d_t.ipynb, Section 5
  STGCN                 -- training_notebooks/train_1000_landmark_models.ipynb, Section 5
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

FRAME_COUNT = 16
N_HAND_LANDMARKS = 21
POSE_SUBSET_IDX = [11, 12, 13, 14, 15, 16]
FACE_SUBSET_IDX = [70, 63, 105, 66, 107, 336, 296, 334, 293, 300]
N_POINTS = N_HAND_LANDMARKS * 2 + len(POSE_SUBSET_IDX) + len(FACE_SUBSET_IDX)   # 58
LANDMARK_VECTOR_DIM = N_POINTS * 2                                              # 116

# Shared Transformer dims (identical in v9 dual-stream and v10 geometry-only)
LANDMARK_EMBED_DIM = 128
CNN_EMBED_DIM = 256
D_MODEL = 256
N_HEADS = 4
N_ENCODER_LAYERS = 3
FFN_DIM = 512
DROPOUT = 0.5


# ---------------------------------------------------------------- shared blocks
class GeometryStream(nn.Module):
    """[coords, frame-to-frame velocity] -> Linear -> ReLU -> LayerNorm."""

    def __init__(self, input_dim=LANDMARK_VECTOR_DIM, embed_dim=LANDMARK_EMBED_DIM):
        super().__init__()
        self.project = nn.Sequential(nn.Linear(input_dim * 2, embed_dim), nn.ReLU(), nn.LayerNorm(embed_dim))

    def forward(self, x):
        velocity = torch.zeros_like(x)
        velocity[:, 1:] = x[:, 1:] - x[:, :-1]
        return self.project(torch.cat([x, velocity], dim=-1))


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=FRAME_COUNT):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


def _temporal_encoder():
    layer = nn.TransformerEncoderLayer(d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=FFN_DIM,
                                       dropout=DROPOUT, batch_first=True)
    return nn.TransformerEncoder(layer, num_layers=N_ENCODER_LAYERS)


# ---------------------------------------------------------------- v10 geometry-only Transformer
class GeometryOnlySLRModel(nn.Module):
    """Input: landmarks (B, 16, 116). Output: logits (B, num_classes)."""

    def __init__(self, num_classes):
        super().__init__()
        self.geometry_stream = GeometryStream()
        self.to_d_model = nn.Linear(LANDMARK_EMBED_DIM, D_MODEL)
        self.pos_encoding = PositionalEncoding(D_MODEL)
        self.temporal_encoder = _temporal_encoder()
        self.word_head = nn.Linear(D_MODEL, num_classes)

    def forward(self, landmarks):
        fused = self.pos_encoding(self.to_d_model(self.geometry_stream(landmarks)))
        return self.word_head(self.temporal_encoder(fused).mean(dim=1))


# ---------------------------------------------------------------- v9 dual-stream (r3d_18 + geometry)
class AppearanceStream(nn.Module):
    """r3d_18 trunk (stem..layer4), spatial avg-pool, temporal interp back to T, Linear 512->256."""

    def __init__(self, embed_dim=CNN_EMBED_DIM):
        super().__init__()
        from torchvision.models.video import r3d_18
        backbone = r3d_18(weights=None)          # trained weights come from the checkpoint
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])
        self.project = nn.Linear(512, embed_dim)

    def forward(self, x):
        B, T, S, C, H, W = x.shape                          # (B, T, streams, C, H, W)
        x = x.permute(0, 2, 3, 1, 4, 5).reshape(B * S, C, T, H, W)
        feats = self.backbone(x).mean(dim=[-1, -2])         # (B*S, 512, T')
        feats = F.interpolate(feats, size=T, mode="linear", align_corners=False).permute(0, 2, 1)
        feats = self.project(feats)
        return feats.view(B, S, T, -1).permute(0, 2, 1, 3).reshape(B, T, -1)


class DualStreamSLRModel(nn.Module):
    """Inputs: appearance (B, 16, 1, 3, 112, 112) Kinetics-normalised, landmarks (B, 16, 116)."""

    def __init__(self, num_classes, num_appearance_streams=1):
        super().__init__()
        self.appearance_stream = AppearanceStream()
        self.geometry_stream = GeometryStream()
        self.fusion = nn.Linear(CNN_EMBED_DIM * num_appearance_streams + LANDMARK_EMBED_DIM, D_MODEL)
        self.pos_encoding = PositionalEncoding(D_MODEL)
        self.temporal_encoder = _temporal_encoder()
        self.word_head = nn.Linear(D_MODEL, num_classes)

    def forward(self, appearance, landmarks):
        fused = torch.cat([self.appearance_stream(appearance), self.geometry_stream(landmarks)], dim=-1)
        fused = self.pos_encoding(self.fusion(fused))
        return self.word_head(self.temporal_encoder(fused).mean(dim=1))


# ---------------------------------------------------------------- Video Swin-T
def build_swin3d_t(num_classes, head_dropout=0.5):
    """Input: (B, 3, 16, 224, 224) Kinetics-normalised."""
    from torchvision.models.video import swin3d_t
    m = swin3d_t(weights=None)
    m.head = nn.Sequential(nn.Dropout(head_dropout), nn.Linear(m.head.in_features, num_classes))
    return m


# ---------------------------------------------------------------- ST-GCN (adaptive graph)
HAND_EDGES = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8), (5, 9), (9, 10), (10, 11),
              (11, 12), (9, 13), (13, 14), (14, 15), (15, 16), (13, 17), (17, 18), (18, 19), (19, 20), (0, 17)]


def to_points(x, packing="interleaved"):
    if packing == "interleaved":
        return x.reshape(*x.shape[:-1], N_POINTS, 2)
    if packing == "planar":
        return x.reshape(*x.shape[:-1], 2, N_POINTS).transpose(-1, -2)
    raise ValueError(packing)


def skeleton_edges():
    e = []
    for off in (0, 21):
        e += [(i + off, j + off) for i, j in HAND_EDGES]
    p = {idx: 42 + k for k, idx in enumerate(POSE_SUBSET_IDX)}
    e += [(p[11], p[12]), (p[11], p[13]), (p[13], p[15]), (p[12], p[14]), (p[14], p[16])]
    f0 = 42 + len(POSE_SUBSET_IDX)
    e += [(f0 + i, f0 + i + 1) for i in range(4)] + [(f0 + 5 + i, f0 + 6 + i) for i in range(4)]
    return e


def normalized_adjacency(V, edges):
    A = torch.eye(V)
    for i, j in edges:
        A[i, j] = A[j, i] = 1.0
    d = A.sum(1).pow(-0.5)
    return d[:, None] * A * d[None, :]


class AdaptiveGraphConv(nn.Module):
    def __init__(self, cin, cout, A):
        super().__init__()
        self.register_buffer("A", A)
        self.B = nn.Parameter(torch.zeros_like(A))
        self.conv = nn.Conv2d(cin, cout, 1)

    def forward(self, x):
        return torch.einsum("nctv,vw->nctw", self.conv(x), self.A + self.B)


class STGCNBlock(nn.Module):
    def __init__(self, cin, cout, A, stride=1, dropout=0.1, kt=5):
        super().__init__()
        self.gcn = AdaptiveGraphConv(cin, cout, A)
        self.bn1 = nn.BatchNorm2d(cout)
        self.tcn = nn.Sequential(nn.Conv2d(cout, cout, (kt, 1), (stride, 1), ((kt - 1) // 2, 0)),
                                 nn.BatchNorm2d(cout), nn.Dropout(dropout))
        if cin == cout and stride == 1:
            self.res = nn.Identity()
        else:
            self.res = nn.Sequential(nn.Conv2d(cin, cout, 1, (stride, 1)), nn.BatchNorm2d(cout))

    def forward(self, x):
        return F.relu(self.tcn(F.relu(self.bn1(self.gcn(x)))) + self.res(x))


class STGCN(nn.Module):
    """Input: landmarks (B, 16, 116) -- same tensor as the v10 model."""

    def __init__(self, num_classes, packing="interleaved", in_ch=4, dropout=0.5):
        super().__init__()
        self.packing = packing
        A = normalized_adjacency(N_POINTS, skeleton_edges())
        self.data_bn = nn.BatchNorm1d(in_ch * N_POINTS)
        cfg = [(in_ch, 64, 1), (64, 64, 1), (64, 64, 1), (64, 128, 2), (128, 128, 1), (128, 256, 2), (256, 256, 1)]
        self.blocks = nn.ModuleList(STGCNBlock(i, o, A, s) for i, o, s in cfg)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, landmarks):
        N, T, _ = landmarks.shape
        pts = to_points(landmarks, self.packing)
        vel = torch.zeros_like(pts)
        vel[:, 1:] = pts[:, 1:] - pts[:, :-1]
        x = torch.cat([pts, vel], -1).permute(0, 2, 3, 1).reshape(N, N_POINTS * 4, T)
        x = self.data_bn(x).reshape(N, N_POINTS, 4, T).permute(0, 2, 3, 1)
        for blk in self.blocks:
            x = blk(x)
        return self.fc(self.drop(x.mean(dim=(2, 3))))

## 4. (A) R3D-18 dual-stream -- test evaluation + export

In [ ]:
def summarize_predictions(y_true, y_pred, top5, num_classes):
    labels = list(range(num_classes))
    mp, mr, mf, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    wp, wr, wf, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average="weighted", zero_division=0)
    return {
        "top1": float((y_true == y_pred).mean()),
        "top5": float(np.mean([t in row for t, row in zip(y_true, top5)])),
        "macro_precision": float(mp), "macro_recall": float(mr), "macro_f1": float(mf),
        "weighted_precision": float(wp), "weighted_recall": float(wr), "weighted_f1": float(wf),
        "n_samples": int(len(y_true)),
    }


def save_evaluation(out_dir, tag, preds, video_ids, class_names):
    """Writes metrics-ready artifacts the web app reads directly."""
    out_dir.mkdir(parents=True, exist_ok=True)
    y_true, y_pred = preds["y_true"], preds["y_pred"]
    n = len(class_names)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n)))
    np.savez_compressed(out_dir / "confusion_matrix.npz", cm=cm.astype(np.uint16))

    p, r, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=list(range(n)), zero_division=0)
    pd.DataFrame({"class_id": range(n), "class_name": class_names, "support": sup,
                  "correct": np.diag(cm), "precision": p, "recall": r, "f1": f1}
                 ).to_csv(out_dir / "per_class_metrics.csv", index=False)

    pd.DataFrame({"video_id": video_ids, "y_true": y_true, "y_pred": y_pred,
                  "confidence": preds["confidence"],
                  "top5": [" ".join(map(str, t)) for t in preds["top5"]]}
                 ).to_csv(out_dir / "test_predictions.csv", index=False)

    pairs = [(class_names[t], class_names[q], int(cm[t, q])) for t, q in zip(*np.nonzero(cm)) if t != q]
    pairs_df = pd.DataFrame(pairs, columns=["true", "pred", "errors"]).sort_values("errors", ascending=False)
    pairs_df.head(50).to_csv(out_dir / "top_confusion_pairs.csv", index=False)
    print(f"[{tag}] top confused pairs:")
    print(pairs_df.head(10).to_string(index=False))
    return cm


def plot_history(history, title, path):
    if not history:
        return
    ep = [h["epoch"] for h in history]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(ep, [h["train_loss"] for h in history], label="train")
    ax[0].plot(ep, [h["val_loss"] for h in history], label="val")
    ax[0].set_title(f"{title} -- loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(ep, [h["train_top1"] for h in history], label="train top-1")
    ax[1].plot(ep, [h["val_top1"] for h in history], label="val top-1")
    ax[1].plot(ep, [h["val_top5"] for h in history], label="val top-5", linestyle="--")
    ax[1].set_title(f"{title} -- accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.savefig(path, dpi=120, bbox_inches="tight"); plt.show()


def measure_cpu_latency(model_cpu, example, n_runs=3, threads=2):
    """Single-clip CPU latency, roughly what a free 2-vCPU host sees (indicative only)."""
    prev = torch.get_num_threads()
    torch.set_num_threads(threads)
    model_cpu.eval()
    with torch.inference_mode():
        model_cpu(example)  # warm-up
        times = []
        for _ in range(n_runs):
            t0 = time.perf_counter(); model_cpu(example); times.append(time.perf_counter() - t0)
    torch.set_num_threads(prev)
    return {"cpu_threads": threads, "mean_s": float(np.mean(times)), "runs": n_runs}


KINETICS_MEAN = torch.tensor([0.43216, 0.394666, 0.37645]).view(1, 1, 3, 1, 1)
KINETICS_STD = torch.tensor([0.22803, 0.22145, 0.216989]).view(1, 1, 3, 1, 1)


class DualStreamEvalDataset(Dataset):
    # Exact eval-time __getitem__ of the r3d_18 notebook (augment=False).
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        vid, label = self.samples[i]
        with np.load(cache_index[vid]) as d:
            appearance = torch.from_numpy(d["appearance"]).float() / 255.0
            landmarks = torch.from_numpy(d["landmarks"]).float()
        appearance = appearance.permute(0, 1, 4, 2, 3)                 # T, S, C, H, W
        appearance = (appearance - KINETICS_MEAN) / KINETICS_STD
        landmarks = torch.nan_to_num(landmarks, nan=0.0, posinf=0.0, neginf=0.0)
        return appearance, landmarks, label


ckpt_path = CONFIG.R3D_CHECKPOINT or (os.environ.get("ASL_R3D_CKPT") and Path(os.environ["ASL_R3D_CKPT"]))
if not ckpt_path:
    hits = sorted(glob.glob("/kaggle/input/**/pilot_1000cls/best_model.pt", recursive=True))
    assert hits, "r3d_18 best_model.pt not found -- attach the previous r3d_18 notebook's output."
    ckpt_path = Path(hits[0])
print("R3D checkpoint:", ckpt_path)
ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
r3d = DualStreamSLRModel(NUM_CLASSES)
r3d.load_state_dict(ck["model_state_dict"], strict=True)
r3d = r3d.to(DEVICE).eval()
print(f"loaded epoch {ck['epoch']} (val_top1 {ck['val_top1']:.3f}) -- strict load OK")

loader = DataLoader(DualStreamEvalDataset(test_samples), batch_size=CONFIG.BATCH_SIZE, shuffle=False,
                    num_workers=CONFIG.NUM_WORKERS, pin_memory=DEVICE == "cuda")
yt, yp, conf, top5 = [], [], [], []
t0 = time.time()
with torch.inference_mode():
    for app, lm, y in loader:
        probs = torch.softmax(r3d(app.to(DEVICE), lm.to(DEVICE)).float(), 1)
        c, p = probs.max(1)
        yt.append(y.numpy()); yp.append(p.cpu().numpy()); conf.append(c.cpu().numpy())
        top5.append(probs.topk(min(5, NUM_CLASSES), 1).indices.cpu().numpy())
test_seconds = time.time() - t0
preds = {"y_true": np.concatenate(yt), "y_pred": np.concatenate(yp), "confidence": np.concatenate(conf),
         "top5": np.concatenate(top5)}
test_metrics = summarize_predictions(preds["y_true"], preds["y_pred"], preds["top5"], NUM_CLASSES)
r3d_dir = CONFIG.OUT_DIR / "r3d18_dualstream_1000"
save_evaluation(r3d_dir, "r3d18_dualstream", preds, [v for v, _ in test_samples], CLASS_NAMES)
print(f"\nR3D-18 dual-stream TEST: top-1 {test_metrics['top1']:.3f} (recorded {CONFIG.RECORDED_TEST_TOP1}) | "
      f"top-5 {test_metrics['top5']:.3f} | macro-F1 {test_metrics['macro_f1']:.3f} | {test_seconds / 60:.1f} min")
if not CONFIG.SMOKE_TEST and abs(test_metrics["top1"] - CONFIG.RECORDED_TEST_TOP1) > 0.006:
    print("WARNING: reproduced top-1 differs from the recorded value by more than rounding -- investigate.")

export = r3d_dir / "app_export"
export.mkdir(parents=True, exist_ok=True)
state = {k: v.detach().cpu() for k, v in r3d.state_dict().items()}
torch.save(state, export / "model_weights.pt")
cpu_model = DualStreamSLRModel(NUM_CLASSES)
cpu_model.load_state_dict(state, strict=True)
app_ex, lm_ex, _ = DualStreamEvalDataset(test_samples[:1])[0]
prev =torch.get_num_threads(); torch.set_num_threads(2); cpu_model.eval()
with torch.inference_mode():
    cpu_model(app_ex[None], lm_ex[None]); ts = []
    for _ in range(3):
        t1 = time.perf_counter(); cpu_model(app_ex[None], lm_ex[None]); ts.append(time.perf_counter() - t1)
torch.set_num_threads(prev)
latency = {"cpu_threads": 2, "mean_s": float(np.mean(ts)), "runs": 3}
print(f"CPU latency (2 threads, 1 clip): {latency['mean_s']:.2f} s")

metrics = {"model_id": "r3d18_dualstream_1000", "num_classes": NUM_CLASSES, "class_fingerprint": CLASS_FINGERPRINT,
           "best_epoch": int(ck["epoch"]), "val_top1_best": float(ck["val_top1"]),
           "selection": "best val_loss (v9 notebook)", "test": test_metrics,
           "trainable_params_during_training": 27007080, "total_params": int(sum(p.numel() for p in state.values() if p.is_floating_point())),
           "gpu": torch.cuda.get_device_name(0) if N_GPUS else "cpu", "gpu_test_seconds_total": round(test_seconds, 1),
           "cpu_latency": latency, "smoke_test": CONFIG.SMOKE_TEST}
model_config = {"model_id": "r3d18_dualstream_1000", "display_name": "R3D-18 Dual-Stream (RGB + landmarks)",
                "architecture": "DualStreamSLRModel (v9): r3d_18 appearance stream + geometry stream -> Transformer",
                "num_classes": NUM_CLASSES, "class_fingerprint": CLASS_FINGERPRINT,
                "inputs": {"appearance": "B,T,1,3,112,112 -- cached signer crop /255, Kinetics mean/std",
                           "landmarks": "B,T,116 -- cached MediaPipe vector, NaN -> 0"},
                "frame_count": 16, "img_size": 112,
                "mean": [0.43216, 0.394666, 0.37645], "std": [0.22803, 0.22145, 0.216989]}
for name, obj in [("metrics.json", metrics), ("model_config.json", model_config)]:
    with open(export / name, "w") as f:
        json.dump(obj, f, indent=1)
for name in ["per_class_metrics.csv", "confusion_matrix.npz"]:
    (export / name).write_bytes((r3d_dir / name).read_bytes())
(export / "class_list.json").write_bytes((CONFIG.OUT_DIR / "class_list.json").read_bytes())
del r3d, cpu_model

## 5. (B) Parity sample pack
`CONFIG.N_SAMPLE_VIDEOS` test-split videos from distinct classes (fixed seed), each copied with its
cached `.npz`, plus `samples.csv` (video id, file, gloss, 1000-class index).

In [ ]:
import shutil

video_files = {}
with open(CONFIG.SPLITS_DIR / "test.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        video_files[Path(row["Video file"]).stem] = row["Video file"]

pack = CONFIG.OUT_DIR / "parity_samples"
pack.mkdir(parents=True, exist_ok=True)
rng = random.Random(CONFIG.SEED)
pool = test_samples[:]
rng.shuffle(pool)
chosen, seen = [], set()
for vid, label in pool:
    if label in seen or vid not in video_files or not (VIDEOS_DIR / video_files[vid]).exists():
        continue
    chosen.append((vid, label)); seen.add(label)
    if len(chosen) >= CONFIG.N_SAMPLE_VIDEOS:
        break

rows = []
for vid, label in chosen:
    src = VIDEOS_DIR / video_files[vid]
    shutil.copy2(src, pack / src.name)
    shutil.copy2(cache_index[vid], pack / f"{vid}.npz")
    rows.append({"video_id": vid, "video_file": src.name, "gloss": CLASS_NAMES[label], "class_idx_1000": label})
pd.DataFrame(rows).to_csv(pack / "samples.csv", index=False)
size_mb = sum(p.stat().st_size for p in pack.iterdir()) / 1e6
print(f"Parity pack: {len(rows)} videos + npz ({size_mb:.1f} MB) -> {pack}")
print(pd.DataFrame(rows).head(10).to_string(index=False))